# MirrorTopology Step 1 — Phase D4C-1：4 family partial の **combiner**（封印較正 record）launcher（notebook v0.1）
4 つの PASS した family partial run（E1／E2／E7／E8；同じ commit・campaign・target commitment・登録 pseudo 列・W₂ context）の出力 directory（各 `run_<attempt>`：`d4c1_partial_<F>_run.json`・`d4c1_partial_<F>_record.json`・`archive/`）を Drive から読み，`step1_engine.d4c1_partial`：archive の byte 同一 merge → 各 partial の再検証（`verify_partial_record`：position source／plan／12 位置 Result／per-pseudo eligibility の再導出）→ `combine_family_partials`（共通 identity（engine・source binding・mode・registry／manifest・context／asset・twelve asset・pseudo 列 SHA と順序・commitment・campaign）と family identity の照合；pseudo ごとに全 family の eligible truth を `any_family_truth` で集約；**分母 n＝2000 固定**の WilsonUpper(c+u, n) を登録閾値（support 0.05／strong 0.01）と比較；run reader による再検証）→ `d4c1_sealed_calibration.json`（kind `sealed_calibration`；単一 process の `calibrate_sealed` と provenance 以外は内容同一）→ `load_combined_sealed_record` の往復。usable の値は記録されるが，**本 notebook の PASS は技術的受入れであり較正の承認ではない**（実行後監査・D4C-3 登録が別途）。target・nonce・label は扱わない。標準 runtime で可（配列は読まない）。

**attempt cell の規約**は partial notebook と同一（anchor・1 つの検査関数を staging 前と起動直前の 2 回・attempt 別 initial 非成功 record・信頼 REQUIRED inventory（`REQUIRED_COMBINE` を AST 抽出）・published evidence の内容束縛）。`combine_pass` は rc 0 ∧ `D4C1_COMBINE_PASS` ∧ shape 正常 ∧ 全 gate True ∧ attempt／lock／source／commitment／campaign の束縛 ∧ sealed record の SHA／bytes 一致と内容束縛（kind・commitment・pseudo n 2000＋列 SHA＝登録定数・context SHA＝pins・4 family provenance＝入力 partial の SHA・`all_registered_families`）のときだけ True。


In [ ]:
# --- 0. OUTER LOCK (the only editable cell)
REPO_URL = 'https://github.com/tsujikeita/mirror-topology.git'
REPO_COMMIT = '<full 40-hex commit: the SAME commit the four partial runs executed>'
EXPECTED_INVENTORY_SHA256 = '<sha256 of engine/phaseB/B2_completion_inventory.json inside that commit>'
PARTIAL_RUN_DIRS = {'E1': '/content/drive/MyDrive/MirrorTopology_D4C1/E1/run_<attempt>',      # the run_<attempt> directories of the four PASSed partial runs (holding d4c1_partial_<F>_run.json, d4c1_partial_<F>_record.json, archive/)
                    'E2': '/content/drive/MyDrive/MirrorTopology_D4C1/E2/run_<attempt>',
                    'E7': '/content/drive/MyDrive/MirrorTopology_D4C1/E7/run_<attempt>',
                    'E8': '/content/drive/MyDrive/MirrorTopology_D4C1/E8/run_<attempt>'}
TARGET_COMMITMENT = '<the same 64-hex commitment given to the four partial runs>'
CAMPAIGN_ID = '<the same campaign identity given to the four partial runs>'
STAGE_LOCAL = True                 # copy the four partial run directories once to STAGE_ROOT (disk capacity checked first)
STAGE_ROOT = '/content/stage'


In [ ]:
# --- 1. fresh scratch checkout; inventory + script + pins + registered ledgers / receipts / assets bound; the four partial run directories exist and are PASSed formal runs of this commit / commitment / campaign; the LOCK
import subprocess, sys, os, json, hashlib, shutil, time, re
sha=lambda p: hashlib.sha256(open(p,'rb').read()).hexdigest()
assert re.fullmatch(r'[0-9a-f]{40}', REPO_COMMIT) and re.fullmatch(r'[0-9a-f]{64}', EXPECTED_INVENTORY_SHA256) and re.fullmatch(r'[0-9a-f]{64}', TARGET_COMMITMENT) and isinstance(CAMPAIGN_ID, str) and len(CAMPAIGN_ID)>=8 and '<' not in CAMPAIGN_ID and set(PARTIAL_RUN_DIRS)=={'E1','E2','E7','E8'}
from google.colab import drive; drive.mount('/content/drive')
PARTIAL_RUNS={}
for f,p in PARTIAL_RUN_DIRS.items():
    r=json.load(open(f'{p}/d4c1_partial_{f}_run.json')); assert os.path.isfile(f'{p}/d4c1_partial_{f}_record.json') and os.path.isfile(f'{p}/archive/index.json'), p
    assert r.get('D4C1_PARTIAL_PASS') is True and r.get('family')==f and r.get('probe') is False and r.get('selftest') is False and r.get('target_commitment')==TARGET_COMMITMENT and r.get('campaign_id')==CAMPAIGN_ID, (f, r.get('D4C1_PARTIAL_PASS'), r.get('target_commitment'), r.get('campaign_id'))
    PARTIAL_RUNS[f]=dict(dir=p, run_sha256=sha(f'{p}/d4c1_partial_{f}_run.json'), record_sha256=sha(f'{p}/d4c1_partial_{f}_record.json'), partial_sha256=(r.get('partial') or {}).get('partial_sha256'), attempt=r.get('attempt'), source=r.get('source'))
SCRATCH='/content/d4c1_scratch'; shutil.rmtree(SCRATCH, ignore_errors=True); subprocess.run(['git','clone','-q',REPO_URL,SCRATCH],check=True); subprocess.run(['git','-C',SCRATCH,'checkout','-q',REPO_COMMIT],check=True)
assert subprocess.check_output(['git','-C',SCRATCH,'rev-parse','HEAD']).decode().strip()==REPO_COMMIT and subprocess.check_output(['git','-C',SCRATCH,'status','--porcelain']).decode().strip()==''
MT=SCRATCH; PHASEB=f'{SCRATCH}/engine/phaseB'; PHASEC=f'{SCRATCH}/phaseC'; INV=f'{PHASEB}/B2_completion_inventory.json'; assert sha(INV)==EXPECTED_INVENTORY_SHA256; inv=json.load(open(INV)); SCRIPT=f'{PHASEB}/d/d4c1_calibration.py'; assert sha(SCRIPT)==inv['d_sha256']['d/d4c1_calibration.py']
for f,r in PARTIAL_RUNS.items(): assert (r['source'] or {}).get('script_sha256')==sha(SCRIPT) and (r['source'] or {}).get('inventory_sha256')==sha(INV), f'partial {f} was not produced by this commit'
RA={k: f'{PHASEB}/{k}' for k in ('registered_assets/d2/d2_generation_ledger.json','registered_assets/d3b/d3b_generation_ledger.json','registered_assets/d3b/d3b_outer_receipt.json','registered_assets/d3c/d3c_profile_ledger.json','registered_assets/d3c/d3c_outer_receipt.json','registered_assets/d4c0/d4c0_ledger.json','registered_assets/d4c0/d4c0_outer_receipt.json','registered_assets/b3_2_twelve_assets.json','registered_assets/b3_2_shared_null_asset.json')}
PINS=f'{PHASEB}/d/d3_pins.json'; assert sha(PINS)==inv['d_sha256']['d/d3_pins.json'] and all(sha(p)==inv['registered_assets_sha256'][k] for k,p in RA.items()); RA_SHA={k: sha(p) for k,p in RA.items()}
pins=json.load(open(PINS)); ex=pins['environment']; subprocess.run([sys.executable,'-m','pip','install','-q',f"numpy=={ex['numpy']}",f"scipy=={ex['scipy']}",f"healpy=={ex['healpy']}",f"camb=={ex['camb']}",f"pot=={ex['pot']}",'threadpoolctl','psutil'],check=True)
os.environ['OPENBLAS_NUM_THREADS']='2'; os.environ['OMP_NUM_THREADS']='2'
import platform, importlib; live={k: importlib.import_module({'pot':'ot'}.get(k,k)).__version__ for k in ('numpy','scipy','healpy','camb','pot')}; live['python']=platform.python_version(); mism={k:(live[k],ex[k]) for k in live if live[k]!=ex[k]}; assert not mism, f'environment lock failed (launcher pre-check): {mism}'
import psutil; vm=psutil.virtual_memory(); RAM_INFO=dict(total_bytes=vm.total, available_bytes=vm.available); print('RAM total GB', round(vm.total/1e9,1), 'available GB', round(vm.available/1e9,1))
OUT='/content/d4c1_combine'
# OUTPUT-ANCHOR (R-D4C0-B; the lines from here to the end of this cell are re-used verbatim by the contract test): the output directory is validated BEFORE the first mkdir / lock publication
_in=lambda p,q: os.path.realpath(p)==os.path.realpath(q) or os.path.realpath(p).startswith(os.path.realpath(q).rstrip('/')+'/')
OWNED=re.compile(r'(d4c1_combine_lock\.json|d4c1_combine_final_record\.json|superseded_final_record_before_[0-9TZ]+_[0-9a-f]{10}\.json|run_[0-9TZ]+_[0-9a-f]{10}|d4c1_combine_(stdout|stderr)_[0-9TZ]+_[0-9a-f]{10}\.txt)')
def _protected_roots(): return [MT, PHASEB, PHASEC, STAGE_ROOT, '/content/drive', *PARTIAL_RUN_DIRS.values()]
def _safe_output_anchor(path, protected):
    # the ONE output policy (preflight and attempt): absolute path whose final component is not a symlink; disjoint from every protected root (source, Phase C, stage, Drive, inputs);
    # non-existent, or an existing real directory holding ONLY files owned by this launcher (lock, attempt records, run_* directories, stdout/stderr) and no symlinks. Nothing is written here.
    ap=os.path.abspath(path)
    if os.path.islink(ap) or os.path.realpath(ap)!=os.path.join(os.path.realpath(os.path.dirname(ap)), os.path.basename(ap)): raise RuntimeError(f'unsafe output: {path!r} is a symbolic link; nothing written')
    clash=[q for q in protected if _in(ap,q) or _in(q,ap)]
    if clash: raise RuntimeError(f'unsafe output: {path!r} overlaps a protected root {clash}; nothing written')
    if os.path.lexists(ap):
        if not os.path.isdir(ap): raise RuntimeError(f'unsafe output: {path!r} exists and is not a directory; nothing written')
        foreign=[e for e in sorted(os.listdir(ap)) if not OWNED.fullmatch(e) or os.path.islink(os.path.join(ap,e))]
        if foreign: raise RuntimeError(f'unsafe output: {path!r} holds entries not owned by this launcher {foreign[:5]}; nothing written')
    return ap
OUT=_safe_output_anchor(OUT, _protected_roots()); os.makedirs(OUT, exist_ok=True)      # attempts are separated inside OUT by their attempt id (nothing is deleted here)
# LOCK-BUILD
def _publish(path, doc):
    data=json.dumps(doc, indent=1, ensure_ascii=False, default=str).encode('utf-8'); tmp=path+'.tmp'; open(tmp,'wb').write(data); os.replace(tmp, path); assert open(path,'rb').read()==data, path; return hashlib.sha256(data).hexdigest()
lock=dict(schema='d4c1_combine_launcher_lock_v1', commit=REPO_COMMIT, inventory_sha256=sha(INV), script_sha256=sha(SCRIPT), inventory_script_sha256=inv['d_sha256']['d/d4c1_calibration.py'], pins_sha256=sha(PINS), registered_assets_sha256=RA_SHA, engine=inv['engine_version'],
          partial_run_dirs=dict(PARTIAL_RUN_DIRS), partial_runs=PARTIAL_RUNS, target_commitment=TARGET_COMMITMENT, campaign_id=CAMPAIGN_ID, stage_local=bool(STAGE_LOCAL), stage_root=STAGE_ROOT, mt=MT, phaseb=PHASEB, phasec=PHASEC, script=SCRIPT, out=OUT, locked_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()))
LOCK_PATH=f'{OUT}/d4c1_combine_lock.json'; LOCK_SHA256=_publish(LOCK_PATH, lock); print('lock', LOCK_SHA256); print({k: v for k, v in lock.items() if k not in ('registered_assets_sha256','partial_runs')})


In [ ]:
# --- 2. ONE ATTEMPT (same pattern as the partial launcher): (0) anchored output from the LOCK validated BEFORE anything is written; (1) attempt id + initial NON-success record; (2) ONE verifier BEFORE staging;
#        (3) staging of the four partial run directories (disk check first); (4) the SAME verifier IMMEDIATELY before subprocess.run; (5) record validation (trusted REQUIRED_COMBINE) + binding + published sealed record content AND semantic check
t0=time.time(); _in=lambda p,q: os.path.realpath(p)==os.path.realpath(q) or os.path.realpath(p).startswith(os.path.realpath(q).rstrip('/')+'/')
ANCHOR=lock['out']
try:
    if '_safe_output_anchor' not in globals(): raise RuntimeError('preflight cell not run')
    _safe_output_anchor(ANCHOR, [lock['mt'], lock['phaseb'], lock['phasec'], lock['stage_root'], '/content/drive', *lock['partial_run_dirs'].values(), *_protected_roots()])
    if not os.path.isdir(ANCHOR) or LOCK_PATH!=f'{ANCHOR}/d4c1_combine_lock.json' or not os.path.isfile(LOCK_PATH): raise RuntimeError('lock path not inside the anchor')
except Exception as _anchor_error: print(f'ATTEMPT NOT STARTED: the locked output {ANCHOR!r} is not a safe anchored directory ({_anchor_error}); nothing was written', file=sys.stderr); raise RuntimeError('unsafe output anchor; nothing written')
ATTEMPT=time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())+'_'+hashlib.sha256(f'{time.time_ns()}|D4C1C|{REPO_COMMIT}|{LOCK_SHA256}'.encode()).hexdigest()[:10]
FIN=f'{ANCHOR}/d4c1_combine_final_record.json'; RUN=f'{ANCHOR}/run_{ATTEMPT}'; combine_pass=False; rc=None; v=None
REC=dict(schema='d4c1_combine_launcher_final_record_v1', attempt_id=ATTEMPT, started_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), lock_sha256=LOCK_SHA256, lock=lock, output_anchor=ANCHOR, ram=globals().get('RAM_INFO'), stage='started', combine_pass=False, exit_code=None, D4C1_COMBINE_PASS=None, launcher_fallback=False, gates_ok=None, bindings_ok=None, evidence_ok=None, failures=[], exception=None, run_dir=RUN, staged_inputs=None, bindings={}, seconds_total=None, finished_utc=None)
if os.path.exists(FIN): prev=f'{ANCHOR}/superseded_final_record_before_{ATTEMPT}.json'; os.replace(FIN, prev); REC['superseded_previous_record']=prev
_publish(FIN, REC)
def _finish(stage, failures=(), exc=None):
    REC.update(stage=stage, exception=(repr(exc) if exc is not None else None), seconds_total=round(time.time()-t0, 3), finished_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())); REC['failures']+=list(failures); REC['combine_pass']=bool(combine_pass); _publish(FIN, REC); print('attempt', ATTEMPT, '| stage', stage, '| combine_pass', REC['combine_pass'], '| failures', REC['failures'])
def _verify_locked_state(prefix):
    REC['stage']=f'{prefix}_config'; cfg=dict(partial_run_dirs=dict(PARTIAL_RUN_DIRS), target_commitment=TARGET_COMMITMENT, campaign_id=CAMPAIGN_ID, stage_local=bool(STAGE_LOCAL), stage_root=STAGE_ROOT, commit=REPO_COMMIT, mt=MT, phaseb=PHASEB, phasec=PHASEC, script=SCRIPT, out=OUT)
    mism={k: (cfg[k], lock.get(k)) for k in cfg if cfg[k]!=lock.get(k)}
    if mism: raise RuntimeError(f'configuration differs from the lock (re-run the preflight cell): {mism}')
    if LOCK_PATH!=f'{ANCHOR}/d4c1_combine_lock.json' or sha(LOCK_PATH)!=LOCK_SHA256 or json.load(open(LOCK_PATH))!=lock: raise RuntimeError('lock file on disk differs from the lock in memory')
    REC['stage']=f'{prefix}_source'; inv_now=json.load(open(INV))
    src=dict(head=subprocess.check_output(['git','-C',MT,'rev-parse','HEAD'],text=True).strip(), clean=(subprocess.check_output(['git','-C',MT,'status','--porcelain'],text=True).strip()==''), inventory_sha256=sha(INV), script_sha256=sha(SCRIPT), inventory_script_sha256=inv_now['d_sha256']['d/d4c1_calibration.py'], pins_sha256=sha(PINS), registered_assets_sha256={k: sha(p) for k,p in RA.items()}, engine=inv_now['engine_version'], partial_records={f: sha(f"{p}/d4c1_partial_{f}_record.json") for f,p in PARTIAL_RUN_DIRS.items()}, checked_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()))
    want=dict(head=lock['commit'], clean=True, inventory_sha256=lock['inventory_sha256'], script_sha256=lock['script_sha256'], inventory_script_sha256=lock['script_sha256'], pins_sha256=lock['pins_sha256'], registered_assets_sha256=lock['registered_assets_sha256'], engine=lock['engine'], partial_records={f: r['record_sha256'] for f,r in lock['partial_runs'].items()})
    bad=[k for k in want if src[k]!=want[k]]; REC['bindings'][f'live_source_{prefix}']=src
    if bad or PHASEB!=f'{MT}/engine/phaseB' or PHASEC!=f'{MT}/phaseC' or SCRIPT!=f'{PHASEB}/d/d4c1_calibration.py': raise RuntimeError(f'live source / partial records differ from the lock ({prefix}): {bad}')
try:
    _verify_locked_state('precheck'); _publish(FIN, REC)
    REC['stage']='staging'; IN=dict(PARTIAL_RUN_DIRS)
    if STAGE_LOCAL:
        ST=STAGE_ROOT; guard=[q for q in [*PARTIAL_RUN_DIRS.values(), ANCHOR, MT] if _in(ST,q) or _in(q,ST)]
        if guard: raise RuntimeError(f'STAGE_ROOT must be disjoint from the inputs, the output and the source: {guard}')
        dsize=lambda root: sum(os.path.getsize(os.path.join(d,x)) for d,_,fs in os.walk(root) for x in fs); need=sum(dsize(p) for p in PARTIAL_RUN_DIRS.values()); ts=time.time()
        shutil.rmtree(ST, ignore_errors=True); os.makedirs(ST); free=shutil.disk_usage(ST).free; REC['staging']=dict(root=ST, input_bytes=need, disk_free_bytes_before=free)
        if free < need*2.1+2e9: raise RuntimeError(f'insufficient local disk for staging + merged archive: need {2*need/1e9:.1f} GB (+5% +2 GB margin), free {free/1e9:.1f} GB')
        IN={f: f'{ST}/partial_{f}' for f in PARTIAL_RUN_DIRS}
        for f,p in PARTIAL_RUN_DIRS.items(): shutil.copytree(p, IN[f])
        REC['staging'].update(seconds=round(time.time()-ts,1), disk_free_bytes_after=shutil.disk_usage(ST).free); print('staged in', round(time.time()-ts), 's')
    REC['staged_inputs']=dict(IN); _publish(FIN, REC)
    _verify_locked_state('prelaunch')
    REC['stage']='launch'; args=[sys.executable,SCRIPT,'--mt',MT,'--phaseb',PHASEB,'--phasec',PHASEC,'--out',RUN,'--mode','combine','--target-commitment',TARGET_COMMITMENT,'--campaign-id',CAMPAIGN_ID,'--profile','production_official','--attempt-id',ATTEMPT,'--launcher-lock-sha256',LOCK_SHA256]+[x for f in sorted(IN) for x in ('--partial', f'{f}={IN[f]}')]
    REC['args']=args; _publish(FIN, REC); rc=subprocess.run(args,capture_output=True,text=True); REC['exit_code']=rc.returncode
    open(f'{ANCHOR}/d4c1_combine_stdout_{ATTEMPT}.txt','w').write(rc.stdout); open(f'{ANCHOR}/d4c1_combine_stderr_{ATTEMPT}.txt','w').write(rc.stderr); print(rc.stdout[-3000:]); print('rc', rc.returncode)
    REC['stage']='record'; vp=f'{RUN}/d4c1_combine_run.json'; shape=[]; gates_bad=[]; binding=[]; evidence=[]
    REQUIRED_EXPECTED=['G_pins_loaded','G_engine_inventory','G_script_sha','G_phaseC_members','G_env_lock','G_external_loader_sha','G_twelve_context','G_w2_context_registered','G_pseudo_registered','G_twelve_asset_registered','G_commitment_form','G_partials_loaded','G_archives_merged','G_partials_verified','G_combined','G_sealed_published','G_sealed_loaded','G_record_saved']
    import ast as _ast
    _req_src=[list(_ast.literal_eval(st.value)) for st in _ast.parse(open(SCRIPT).read()).body if isinstance(st,_ast.Assign) and any(isinstance(t,_ast.Name) and t.id=='REQUIRED_COMBINE' for t in st.targets)]
    if _req_src!=[REQUIRED_EXPECTED]: gates_bad.append(f'REQUIRED_COMBINE inventory of the lock-bound script differs from the trusted constant: {_req_src}')
    try: v=json.load(open(vp))
    except Exception as read_error: v=None; shape.append(f'run record unreadable: {read_error!r}')
    if v is not None and not isinstance(v, dict): shape.append('run record is not a dictionary'); v=None
    if v is not None:
        num=lambda x: isinstance(x,(int,float)) and not isinstance(x,bool)
        for k, ok in [('schema', v.get('schema')=='d4c1_run_record_v1'), ('mode', v.get('mode')=='combine'), ('D4C1_COMBINE_PASS', isinstance(v.get('D4C1_COMBINE_PASS'), bool)), ('stage', v.get('stage')=='complete'), ('gates', isinstance(v.get('gates'), dict)), ('required_inventory', isinstance(v.get('required_inventory'), list)), ('required_all_true', isinstance(v.get('required_all_true'), bool)), ('failures', v.get('failures')==[]), ('seconds', num(v.get('seconds'))),
                      ('attempt', isinstance(v.get('attempt'), dict)), ('source', isinstance(v.get('source'), dict)), ('selftest', v.get('selftest') is False), ('formal', v.get('formal') is True), ('probe', v.get('probe') is False), ('profile', v.get('profile')=='production_official'), ('sealed', isinstance(v.get('sealed'), dict)), ('partials', isinstance(v.get('partials'), dict)), ('pseudo', isinstance(v.get('pseudo'), dict)), ('w2_context', isinstance(v.get('w2_context'), dict)), ('published_evidence', isinstance(v.get('published_evidence'), dict))]:
            if not ok: shape.append(f'run record field invalid: {k}')
        g=v.get('gates') if isinstance(v.get('gates'), dict) else {}
        if sorted(g)!=sorted(REQUIRED_EXPECTED): gates_bad.append(f'gate set differs from the trusted REQUIRED inventory: {sorted(g)}')
        if v.get('required_inventory')!=REQUIRED_EXPECTED: gates_bad.append('required_inventory differs from the trusted REQUIRED inventory')
        if not all(g.get(k) is True for k in REQUIRED_EXPECTED): gates_bad.append(f'required gates not all True: {[k for k in REQUIRED_EXPECTED if g.get(k) is not True]}')
        if v.get('required_all_true') is not True: gates_bad.append('required_all_true is not True')
        a_=v.get('attempt') if isinstance(v.get('attempt'), dict) else {}; s_=v.get('source') if isinstance(v.get('source'), dict) else {}; pr=v.get('partials') if isinstance(v.get('partials'), dict) else {}
        for k, ok in [('attempt_id', a_.get('attempt_id')==ATTEMPT), ('launcher_lock_sha256', a_.get('launcher_lock_sha256')==LOCK_SHA256), ('source.script_sha256', s_.get('script_sha256')==lock['script_sha256']), ('source.inventory_sha256', s_.get('inventory_sha256')==lock['inventory_sha256']), ('source.pins_sha256', s_.get('pins_sha256')==lock['pins_sha256']), ('source.engine_version', s_.get('engine_version')==lock['engine']), ('target_commitment', v.get('target_commitment')==lock['target_commitment']), ('campaign_id', v.get('campaign_id')==lock['campaign_id']),
                      ('partial_records', sorted(pr)==['E1','E2','E7','E8'] and all(isinstance(pr.get(f), dict) and pr[f].get('record_sha256')==lock['partial_runs'][f]['record_sha256'] and pr[f].get('D4C1_PARTIAL_PASS') is True for f in ('E1','E2','E7','E8')))]:
            if not ok: binding.append(f'run record not bound to this attempt / lock / partials: {k}')
        pe=v.get('published_evidence') if isinstance(v.get('published_evidence'), dict) else {}; fn='d4c1_sealed_calibration.json'; fp=f'{RUN}/{fn}'; e=pe.get(fn); ps_=v.get('pseudo') if isinstance(v.get('pseudo'), dict) else {}; idn=ps_.get('identity') if isinstance(ps_.get('identity'), dict) else {}; sealed=v.get('sealed') if isinstance(v.get('sealed'), dict) else {}
        if set(pe)!={fn}: evidence.append(f'published evidence set {sorted(pe)} != {[fn]}')
        if not (isinstance(e, dict) and os.path.isfile(fp) and not os.path.islink(fp) and sha(fp)==e.get('sha256') and os.path.getsize(fp)==e.get('bytes')): evidence.append(f'published sealed record not content-verified: {fn}')
        else:
            try: doc=json.load(open(fp))
            except Exception as doc_error: doc=None; evidence.append(f'published sealed record unreadable: {doc_error!r}')
            if isinstance(doc, dict):
                thr=doc.get('thresholds') if isinstance(doc.get('thresholds'), dict) else {}; pp=thr.get('pseudo') if isinstance(thr.get('pseudo'), dict) else {}; b=doc.get('binding') if isinstance(doc.get('binding'), dict) else {}; comb=b.get('combiner') if isinstance(b.get('combiner'), dict) else {}; cp=comb.get('partials') if isinstance(comb.get('partials'), dict) else {}; bc=doc.get('branch_completeness') if isinstance(doc.get('branch_completeness'), dict) else {}; cal=doc.get('calibration') if isinstance(doc.get('calibration'), dict) else {}
                checks=[('mode', doc.get('mode')=='official'), ('engine', doc.get('engine_version')==lock['engine']), ('commitment', thr.get('target') is None and thr.get('target_commitment')==lock['target_commitment']==comb.get('target_commitment')), ('pseudo', pp.get('n')==2000 and pp.get('sha256_T1')==idn.get('T1_sha256') and pp.get('sha256_T2')==idn.get('T2_sha256') and idn.get('paired_sha256')==pins['pseudo_paired_sha256'] and len(doc.get('per_pseudo_family_status') or [])==2000),
                        ('context_sha', doc.get('w2_context_sha256')==pins['d2w_context_sha256']), ('kind', (b.get('run_manifest_ref') or {}).get('identity',{}).get('kind')=='sealed_calibration' and b.get('run_manifest_sha256')==sealed.get('run_manifest_sha256')), ('provenance', sorted(cp)==['E1','E2','E7','E8'] and all(cp[f].get('partial_sha256')==lock['partial_runs'][f]['partial_sha256'] for f in cp) and (comb.get('campaign') or {}).get('id')==lock['campaign_id']),
                        ('all_families', bc.get('all_registered_families') is True and (doc.get('binding') or {}).get('sealed_dependencies',{}).get('families')==['E1','E2','E7','E8'] and (doc.get('binding') or {}).get('sealed_dependencies',{}).get('require_all_families') is True), ('calibration_levels', set(cal)=={'support','strong'} and all(isinstance(cal[l].get('summary'), dict) and cal[l]['summary'].get('n')==2000 and cal[l]['summary'].get('threshold')==(0.05 if l=='support' else 0.01) for l in cal)), ('no_label', doc.get('final_label_released') is False and not doc.get('families') and not doc.get('cases')), ('archive_index', os.path.isfile(f'{RUN}/archive/index.json'))]
                for k, ok in checks:
                    if not ok: evidence.append(f'published sealed record not bound: {k}')
        REC.update(D4C1_COMBINE_PASS=v.get('D4C1_COMBINE_PASS'), record_sha256=sha(vp), record_stage=v.get('stage'), record_failures=v.get('failures'), gates=v.get('gates'), seconds_script=v.get('seconds'), timings=v.get('timings'), sealed_summary={k: (v.get('sealed') or {}).get(k) for k in ('run_manifest_sha256','run_manifest_file_sha256','calibration','full_procedure','branch_completeness')})
        print('rc', rc.returncode, '| stage', v.get('stage'), '| D4C1_COMBINE_PASS', v.get('D4C1_COMBINE_PASS'), '| failures', v.get('failures')); print('sealed', REC['sealed_summary'])
    REC.update(launcher_fallback=bool(shape), gates_ok=(not gates_bad) if v is not None else None, bindings_ok=(not binding) if v is not None else None, evidence_ok=(not evidence) if v is not None else None, stderr_tail=rc.stderr[-2000:])
    combine_pass=bool(rc.returncode==0 and v is not None and v.get('D4C1_COMBINE_PASS') is True and not shape and not gates_bad and not binding and not evidence)      # the value of usable is NOT a condition (technical acceptance only; post-run audit / registration separate)
    _finish('complete', shape+gates_bad+binding+evidence)
except Exception as attempt_error:
    combine_pass=False; _finish(REC.get('stage') or 'unknown', [f'exception at stage {REC.get("stage")}: {attempt_error!r}'], attempt_error)
print('combine_pass', combine_pass)


In [ ]:
# --- 3. zip the evidence (lock, attempt records, run record, sealed calibration record, merged archive, stdout/stderr) for the post-run audit and D4C-3 registration
zp=f'/content/d4c1_combine_{REPO_COMMIT[:12]}.zip'; shutil.make_archive(zp[:-4], 'zip', root_dir=lock['out'])
from google.colab import files; print(zp, os.path.getsize(zp)); files.download(zp)
